# Thử nghiệm và đánh giá kmean khi dùng 3 loại feature khác nhau

In [11]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
)
from sklearn.preprocessing import StandardScaler

In [12]:

# 1. CONFIGURATION

DB_PATH = ("../data/database/career.db")
OUTPUT_DIR = ("../outputs/kmeans_experiments")

RANDOM_STATE = 42
K_VALUES = range(2, 9)
PCA_COMPONENTS = 14

BASELINE_FEATURES = [
    "cognitive_intensity",
    "social_intensity",
    "physical_operational_intensity",
    "information_digital_intensity",
    "administrative_routine_intensity",
]

In [13]:
def load_feature_matrices(db_path):
    with sqlite3.connect(db_path) as conn:
        # Load baseline features
        baseline = pd.read_sql_query(
            """
            SELECT *
            FROM occupation_features
            """,conn
        )

        # Load detail work activities (41 lv)
        work_activities = pd.read_sql_query(
            """SELECT
                occupation_code,
                activity,
                level_score
            FROM occupation_work_activities
            """,conn
        )


    # Validate required columns
    required_baseline = {"occupation_code", *BASELINE_FEATURES}
    missing_columns = required_baseline - set(baseline.columns)

    if missing_columns:
        raise ValueError(
            f"Missing baseline columns: {sorted(missing_columns)}. "
            "Check the actual schema of occupation_features."
        )

    if baseline["occupation_code"].duplicated().any():
        raise ValueError(
            "occupation_features contains duplicate occupation_code."
        )

    if work_activities.duplicated(
        ["occupation_code", "activity"]
    ).any():
        raise ValueError(
            "Duplicate occupation-activity pairs detected."
        )

    # Build 41 LV matrix
    X_41 = work_activities.pivot(
        index="occupation_code",
        columns="activity",
        values="level_score",
    )

    X_41.columns.name = None
    X_41.index.name = "occupation_code"

    if X_41.shape[1] != 41:
        raise ValueError(
            f"Expected 41 activities, got {X_41.shape[1]}."
        )

    if X_41.isna().any().any():
        raise ValueError("Missing values found in 41 LV matrix.")

    # Select baseline columns
    X_5 = baseline.set_index("occupation_code")[
        BASELINE_FEATURES
    ].copy()

    if X_5.isna().any().any():
        raise ValueError("Missing values found in baseline features.")

    # Use the common occupations for fair comparison
    common_codes = X_5.index.intersection(X_41.index).sort_values()

    X_5 = X_5.loc[common_codes]
    X_41 = X_41.loc[common_codes]

    if X_5.shape[0] < 3:
        raise ValueError("Too few common occupations for clustering.")

    if not np.isfinite(X_5.to_numpy()).all():
        raise ValueError("Non-finite values found in baseline features.")

    if not np.isfinite(X_41.to_numpy()).all():
        raise ValueError("Non-finite values found in 41 LV features.")

    print("=== FEATURE MATRICES ===")
    print("Common occupations:", len(common_codes))
    print("Baseline:", X_5.shape)
    print("41 LV:", X_41.shape)

    return X_5, X_41

In [14]:
# =========================================================
# 3. PREPROCESSING
# =========================================================

def prepare_representations(X_5, X_41):
    # Standardize each representation independently
    scaler_5 = StandardScaler()
    X_5_scaled = scaler_5.fit_transform(X_5)

    scaler_41 = StandardScaler()
    X_41_scaled = scaler_41.fit_transform(X_41)

    # PCA is fitted on standardized 41 LV data
    pca = PCA(n_components=PCA_COMPONENTS)
    X_pca = pca.fit_transform(X_41_scaled)

    actual_variance = pca.explained_variance_ratio_.sum()

    print("\n=== PCA ===")
    print("Components:", pca.n_components_)
    print(f"Explained variance: {actual_variance:.4f}")

    return {
        "baseline_5": (X_5.index, X_5_scaled),
        "work_activities_41_lv": (X_41.index, X_41_scaled),
        "work_activities_pca_14": (X_41.index, X_pca),
    }


In [15]:
# =========================================================
# 4. RUN K-MEANS EXPERIMENTS
# =========================================================

def evaluate_clustering(X, labels):
    number_of_clusters = len(np.unique(labels))

    # Silhouette and DB require at least 2 clusters,
    # and fewer clusters than observations.
    if number_of_clusters < 2 or number_of_clusters >= len(labels):
        return {
            "silhouette": np.nan,
            "davies_bouldin": np.nan,
            "calinski_harabasz": np.nan,
        }

    return {
        "silhouette": silhouette_score(X, labels),
        "davies_bouldin": davies_bouldin_score(X, labels),
        "calinski_harabasz": calinski_harabasz_score(X, labels),
    }


def run_experiments(representations):
    results = []
    assignments = {}

    for representation_name, (occupation_codes, X) in representations.items():

        print(f"\n=== {representation_name} ===")

        for k in K_VALUES:
            model = KMeans(
                n_clusters=k,
                init="k-means++",
                n_init=20,
                random_state=RANDOM_STATE,
            )

            labels = model.fit_predict(X)

            metrics = evaluate_clustering(X, labels)

            cluster_sizes = pd.Series(labels).value_counts()

            result = {
                "representation": representation_name,
                "k": k,
                "inertia": model.inertia_,
                "silhouette": metrics["silhouette"],
                "davies_bouldin": metrics["davies_bouldin"],
                "calinski_harabasz": metrics["calinski_harabasz"],
                "smallest_cluster": int(cluster_sizes.min()),
                "largest_cluster": int(cluster_sizes.max()),
                "n_clusters_found": int(len(cluster_sizes)),
            }

            results.append(result)

            assignment_key = (representation_name, k)

            assignments[assignment_key] = pd.DataFrame({
                "occupation_code": occupation_codes.to_numpy(),
                "cluster_id": labels,
            })

            print(
                f"K={k} | "
                f"Silhouette={metrics['silhouette']:.4f} | "
                f"DB={metrics['davies_bouldin']:.4f} | "
                f"Inertia={model.inertia_:.2f} | "
                f"Cluster sizes={sorted(cluster_sizes.tolist())}"
            )

    return pd.DataFrame(results), assignments

In [16]:
# =========================================================
# 5. SAVE RESULTS
# =========================================================

def save_results(results, assignments, output_dir):
    output_dir.mkdir(parents=True, exist_ok=True)

    results = results.sort_values(
        ["representation", "k"]
    )

    results.to_csv(
        output_dir / "kmeans_comparison.csv",
        index=False,
    )

    for (representation, k), assignment in assignments.items():
        filename = f"{representation}_k{k}.csv"

        assignment.to_csv(
            output_dir / filename,
            index=False,
        )

    print("\nResults saved to:", output_dir.resolve())

In [17]:
# =========================================================
# 6. MAIN
# =========================================================

def main():
    X_5, X_41 = load_feature_matrices(DB_PATH)

    representations = prepare_representations(X_5, X_41)

    results, assignments = run_experiments(representations)

    save_results(results, assignments, OUTPUT_DIR)

    print("\n=== TOP RESULTS BY SILHOUETTE ===")
    print(
        results.sort_values(
            "silhouette",
            ascending=False,
        )[
            [
                "representation",
                "k",
                "silhouette",
                "davies_bouldin",
                "calinski_harabasz",
                "smallest_cluster",
                "largest_cluster",
            ]
        ].head(10).to_string(index=False)
    )


if __name__ == "__main__":
    main()

ValueError: Missing values found in baseline features.